# PC1
Desarrollar lo siguiente:
1. Escoger 2 ejercicios no vistos en clase (de internet) que se puedan resolver por Fuerza Bruta y/o Bracktracking.
2. Resolverlos con los métodos de FB y/o BT.
3. Resolver esos ejercicios con CP.

Elaborar un notebook con las soluciones de FB y/o BT; y adicional a ello CP.

Finalmente elaborar un video de máximo 10 minutos explicando los ejercicios y la solución desarrollada.
Deben participar todos los integrantes del equipo.

## 1er ejercicio tipo Placement Puzzles: Puzzles (Codeforces 337A)

### Ejercicios tipo "Placement Puzzles"
La idea en estos tipos de problemas es **minimizar el costo de operaciones** y **encontrar la mejor solución (en este caso la mejor combinación)** ya esto dependiendo de las restricciones dichas que sean planteadas en cada problema.

### Descripción del problema "Puzzles"
Una profesora quiere regalar a sus n estudiantes un rompecabezas a cada uno. En la tienda hay m rompecabezas, cada uno con distinta cantidad de piezas. La profesora quiere que la diferencia entre el rompecabezas con más piezas y el que tenga menos piezas (entre los que ella elija) sea lo más pequeña posible.

En otras palabras:
- Se nos da una lista con las cantidades de piezas de los m rompecabezas.
- Debemos elegir n rompecabezas de esa lista.
- El objetivo es minimizar la diferencia A - B, donde:
  - A: número de piezas del rompecabezas más grande elegido.
  - B: número de piezas del rompecabezas más pequeño elegido.

### Idea de solución
1. Ordenamos la lista de rompecabezas por número de piezas.
2. Consideramos todas las posibles subsecuencias de longitud 𝑛 dentro de esa lista ordenada.
3. Para cada subsecuencia, calculamos la diferencia entre el mayor y el menor.
4. La respuesta será el mínimo de esas diferencias.

Esto funciona porque al ordenar, los rompecabezas más cercanos en tamaño quedan juntos, y basta revisar ventanas consecutivas de tamaño 𝑛.

### Con BF (Brute Force)

In [1]:
import sys
import time

def backtrack(start, chosen, n, m, f, min_diff):
  if len(chosen) == n:
    current_diff = max(chosen) - min(chosen)
    if current_diff < min_diff:
      min_diff = current_diff
    
    return min_diff
  
  # recorrer hasta el último índice posible que permita completar la combinación
  remaining_needed = n - len(chosen)
  
  for i in range(start, m - remaining_needed + 1):
    chosen.append(f[i])
    min_diff = backtrack(i + 1, chosen, n, m, f, min_diff)
    chosen.pop()
    # poda temprana: si encontramos 0 ya no puede mejorar
    if min_diff == 0:
      return 0
  
  return min_diff

In [2]:
try:
  n, m = map(int, input("Ingresa n y m separados por espacio: ").split())
except ValueError:
  print("Error: entrada inválida para n y m.")
  sys.exit(1)

try:
  f = list(map(int, input(f"Ingresa {m} números separados por espacio: ").split()))
except ValueError:
  print("Error: entrada inválida para los números.")
  sys.exit(1)

if len(f) != m:
  print(f"Error: Se esperaban {m} números, pero se recibieron {len(f)}.")
  sys.exit(1)

if n > m or n <= 0:
  print("Error: n debe ser entre 1 y m.")
  sys.exit(1)

min_diff = 10**18

start_time = time.time()
resultado = backtrack(0, [], n, m, f, min_diff)
end_time = time.time()

print("Resultado:", resultado)
print(f"Tiempo de ejecución: {end_time - start_time:.9f} segundos")

Resultado: 5
Tiempo de ejecución: 0.000074148 segundos


### Con CP (Constraint Programming)

In [3]:
import sys
import time
from ortools.sat.python import cp_model

def solve_with_cp(n, m, f):
  if n > m or n <= 0:
    raise ValueError("n debe estar entre 1 y m")
  
  model = cp_model.CpModel()

  # Variables booleanas: x[i] = 1 si el elemento i es seleccionado
  x = [model.NewBoolVar(f"x[{i}]") for i in range(m)]
  
  # Variables para el mínimo y máximo entre los seleccionados
  f_min = min(f)
  f_max = max(f)
  min_val = model.NewIntVar(f_min, f_max, "min_val")
  max_val = model.NewIntVar(f_min, f_max, "max_val")
  
  # Seleccionar exactamente n elementos
  model.Add(sum(x) == n)
  
  # Enforce: si x[i] es True entonces min_val <= f[i] y max_val >= f[i]
  for i in range(m):
    # OnlyEnforceIf acepta literales booleanos
    model.Add(min_val <= f[i]).OnlyEnforceIf(x[i])
    model.Add(max_val >= f[i]).OnlyEnforceIf(x[i])
  
  # Poda lógica: min_val >= mínimo posible y max_val <= máximo posible ya están por dominio
  # Objetivo: minimizar la diferencia max_val - min_val
  diff = model.NewIntVar(0, f_max - f_min, "diff")
  model.Add(diff == max_val - min_val)
  model.Minimize(diff)
  
  # Resolver
  solver = cp_model.CpSolver()
  solver.parameters.max_time_in_seconds = 30.0  # límite opcional
  solver.parameters.num_search_workers = 8

  status = solver.Solve(model)
  
  if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    selected = [f[i] for i in range(m) if solver.Value(x[i]) == 1]
    return {
      "status": "OPTIMAL" if status == cp_model.OPTIMAL else "FEASIBLE",
      "min_diff": solver.Value(diff),
      "min_val": solver.Value(min_val),
      "max_val": solver.Value(max_val),
      "selected_values": selected,
      "selected_indices": [i for i in range(m) if solver.Value(x[i]) == 1]
    }
  else:
    return {"status": "INFEASIBLE_OR_UNKNOWN"}

In [4]:
try:
  n, m = map(int, input("Ingresa n y m separados por espacio: ").split())
except Exception:
  print("Error: entrada inválida para n y m.")
  sys.exit(1)

try:
  f = list(map(int, input(f"Ingresa {m} números separados por espacio: ").split()))
except Exception:
  print("Error: entrada inválida para los números.")
  sys.exit(1)

if len(f) != m:
  print(f"Error: Se esperaban {m} números, pero se recibieron {len(f)}.")
  sys.exit(1)

try:
  start_time = time.time()
  result = solve_with_cp(n, m, f)
  end_time = time.time()
  print(f"Tiempo de ejecución: {end_time - start_time:.9f} segundos")
except ValueError as e:
  print("Error:", e)
  sys.exit(1)

if result["status"] in ("OPTIMAL", "FEASIBLE"):
  print("Estado:", result["status"])
  print("Resultado (mínima diferencia):", result["min_diff"])
  print("Valor mínimo seleccionado:", result["min_val"])
  print("Valor máximo seleccionado:", result["max_val"])
  print("Índices seleccionados:", result["selected_indices"])
  print("Valores seleccionados:", result["selected_values"])
else:
  print("No se encontró solución factible.")

Tiempo de ejecución: 0.090017796 segundos
Estado: OPTIMAL
Resultado (mínima diferencia): 5
Valor mínimo seleccionado: 7
Valor máximo seleccionado: 12
Índices seleccionados: [0, 1, 2, 3]
Valores seleccionados: [10, 12, 10, 7]
